In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    lower,
    current_timestamp,
    row_number,
    lit
)
from pyspark.sql.window import Window

CATALOG = "dbw_ecommerce_om"

BRONZE_SCHEMA = "ecommerce"
SILVER_SCHEMA = "silver"

print("Silver transformation configuration loaded")
print(f"Source: {CATALOG}.{BRONZE_SCHEMA}")
print(f"Target: {CATALOG}.{SILVER_SCHEMA}")

Silver transformation configuration loaded
Source: dbw_ecommerce_om.ecommerce
Target: dbw_ecommerce_om.silver


In [0]:
bronze_customers = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.bronze_customers"
)

bronze_products = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.bronze_products"
)

bronze_orders = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.bronze_orders"
)

bronze_order_items = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.bronze_order_items"
)

bronze_payments = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.bronze_payments"
)

print("All five Bronze tables loaded")

All five Bronze tables loaded


In [0]:
silver_customers = (
    bronze_customers
    .select(
        col("customer_id").cast("int").alias("customer_id"),
        trim(col("name")).alias("name"),
        lower(trim(col("email"))).alias("email"),
        trim(col("city")).alias("city"),
        trim(col("state")).alias("state"),
        col("registration_date").cast("date").alias("registration_date")
    )
    .dropDuplicates(["customer_id"])
)

print(f"Silver customers rows: {silver_customers.count()}")

Silver customers rows: 10000


In [0]:
silver_products = (
    bronze_products
    .select(
        col("product_id").cast("int").alias("product_id"),
        trim(col("product_name")).alias("product_name"),
        trim(col("category")).alias("category"),
        col("price").cast("double").alias("price")
    )
    .dropDuplicates(["product_id"])
)

print(f"Silver products rows: {silver_products.count()}")

Silver products rows: 2000


In [0]:
# Identify invalid orders
orders_invalid = bronze_orders.filter(
    col("customer_id").isNull()
    | col("order_date").isNull()
    | (col("total_amount") <= 0)
    | (~upper(trim(col("status"))).isin("COMPLETED", "PENDING", "CANCELLED"))
)

# Keep invalid records for quarantine
orders_quarantine = (
    orders_invalid
    .withColumn("_quarantine_reason", lit("ORDER_DATA_QUALITY_ERROR"))
    .withColumn("_quarantine_timestamp", current_timestamp())
)

# Keep valid orders
orders_valid = bronze_orders.filter(
    col("customer_id").isNotNull()
    & col("order_date").isNotNull()
    & (col("total_amount") > 0)
    & upper(trim(col("status"))).isin("COMPLETED", "PENDING", "CANCELLED")
)

# Standardize and remove duplicate order IDs
window_orders = Window.partitionBy("order_id").orderBy(
    col("_ingestion_timestamp").desc()
)

silver_orders = (
    orders_valid
    .withColumn("_row_number", row_number().over(window_orders))
    .filter(col("_row_number") == 1)
    .drop("_row_number")
    .select(
        col("order_id").cast("int").alias("order_id"),
        col("customer_id").cast("int").alias("customer_id"),
        col("order_date").cast("date").alias("order_date"),
        upper(trim(col("status"))).alias("status"),
        col("total_amount").cast("double").alias("total_amount")
    )
)

print(f"Valid Silver orders: {silver_orders.count()}")
print(f"Quarantined orders: {orders_quarantine.count()}")

Valid Silver orders: 99600
Quarantined orders: 500


In [0]:
# Identify invalid order items
order_items_invalid = bronze_order_items.filter(
    col("product_id").isNull()
    | (col("quantity") <= 0)
)

# Keep invalid records for quarantine
order_items_quarantine = (
    order_items_invalid
    .withColumn(
        "_quarantine_reason",
        lit("ORDER_ITEM_DATA_QUALITY_ERROR")
    )
    .withColumn(
        "_quarantine_timestamp",
        current_timestamp()
    )
)

# Keep valid order items
order_items_valid = bronze_order_items.filter(
    col("product_id").isNotNull()
    & (col("quantity") > 0)
)

# Standardize order items
silver_order_items = (
    order_items_valid
    .select(
        col("order_item_id").cast("int").alias("order_item_id"),
        col("order_id").cast("int").alias("order_id"),
        col("product_id").cast("int").alias("product_id"),
        col("quantity").cast("int").alias("quantity"),
        col("price").cast("double").alias("price")
    )
    .dropDuplicates(["order_item_id"])
)

print(f"Valid Silver order items: {silver_order_items.count()}")
print(f"Quarantined order items: {order_items_quarantine.count()}")

Valid Silver order items: 199600
Quarantined order items: 400


In [0]:
# Identify invalid payments
payments_invalid = bronze_payments.filter(
    ~upper(trim(col("payment_status"))).isin(
        "PAID",
        "PENDING",
        "FAILED"
    )
)

# Keep invalid records for quarantine
payments_quarantine = (
    payments_invalid
    .withColumn(
        "_quarantine_reason",
        lit("INVALID_PAYMENT_STATUS")
    )
    .withColumn(
        "_quarantine_timestamp",
        current_timestamp()
    )
)

# Keep valid payments
payments_valid = bronze_payments.filter(
    upper(trim(col("payment_status"))).isin(
        "PAID",
        "PENDING",
        "FAILED"
    )
)

# Standardize payment fields
silver_payments = (
    payments_valid
    .select(
        col("payment_id").cast("int").alias("payment_id"),
        col("order_id").cast("int").alias("order_id"),
        trim(col("payment_method")).alias("payment_method"),
        upper(trim(col("payment_status"))).alias("payment_status"),
        col("payment_date").cast("date").alias("payment_date")
    )
    .dropDuplicates(["payment_id"])
)

print(f"Valid Silver payments: {silver_payments.count()}")
print(f"Quarantined payments: {payments_quarantine.count()}")

Valid Silver payments: 99900
Quarantined payments: 100


In [0]:
# Write Silver tables
silver_tables = {
    "customers": silver_customers,
    "products": silver_products,
    "orders": silver_orders,
    "order_items": silver_order_items,
    "payments": silver_payments
}

for table_name, df in silver_tables.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(
            f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
        )
    )
    print(f"Written: {CATALOG}.{SILVER_SCHEMA}.{table_name}")


# Write quarantine tables
quarantine_tables = {
    "orders_quarantine": orders_quarantine,
    "order_items_quarantine": order_items_quarantine,
    "payments_quarantine": payments_quarantine
}

for table_name, df in quarantine_tables.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(
            f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
        )
    )
    print(f"Written: {CATALOG}.{SILVER_SCHEMA}.{table_name}")


# Create Data Quality audit records
dq_records = [
    (
        "orders",
        "NULL_CUSTOMER_ID",
        bronze_orders.filter(col("customer_id").isNull()).count()
    ),
    (
        "orders",
        "NULL_ORDER_DATE",
        bronze_orders.filter(col("order_date").isNull()).count()
    ),
    (
        "orders",
        "NON_POSITIVE_AMOUNT",
        bronze_orders.filter(col("total_amount") <= 0).count()
    ),
    (
        "orders",
        "INVALID_STATUS",
        bronze_orders.filter(
            ~upper(trim(col("status"))).isin(
                "COMPLETED", "PENDING", "CANCELLED"
            )
        ).count()
    ),
    (
        "order_items",
        "NULL_PRODUCT_ID",
        bronze_order_items.filter(col("product_id").isNull()).count()
    ),
    (
        "order_items",
        "NON_POSITIVE_QUANTITY",
        bronze_order_items.filter(col("quantity") <= 0).count()
    ),
    (
        "payments",
        "INVALID_PAYMENT_STATUS",
        bronze_payments.filter(
            ~upper(trim(col("payment_status"))).isin(
                "PAID", "PENDING", "FAILED"
            )
        ).count()
    )
]

dq_audit = spark.createDataFrame(
    dq_records,
    [
        "source_table",
        "rule_name",
        "failed_record_count"
    ]
).withColumn(
    "audit_timestamp",
    current_timestamp()
)

(
    dq_audit.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.{SILVER_SCHEMA}.dq_audit"
    )
)

print("Silver tables written successfully")
print("Quarantine tables written successfully")
print("DQ audit table written successfully")

Written: dbw_ecommerce_om.silver.customers
Written: dbw_ecommerce_om.silver.products
Written: dbw_ecommerce_om.silver.orders
Written: dbw_ecommerce_om.silver.order_items
Written: dbw_ecommerce_om.silver.payments
Written: dbw_ecommerce_om.silver.orders_quarantine
Written: dbw_ecommerce_om.silver.order_items_quarantine
Written: dbw_ecommerce_om.silver.payments_quarantine
Silver tables written successfully
Quarantine tables written successfully
DQ audit table written successfully


In [0]:
# Validate Silver tables

print("========== SILVER VALIDATION ==========")

for table_name in silver_tables:
    df = spark.table(
        f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    )

    print(f"{table_name}: {df.count()} rows")
    print(f"Columns: {df.columns}")
    print("-" * 60)


print("========== QUARANTINE VALIDATION ==========")

for table_name in quarantine_tables:
    df = spark.table(
        f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    )

    print(f"{table_name}: {df.count()} rows")
    print("-" * 60)


print("========== DQ AUDIT ==========")

spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.dq_audit"
).show(truncate=False)

print("Silver validation completed successfully")

========== SILVER VALIDATION ==========
customers: 10000 rows
Columns: ['customer_id', 'name', 'email', 'city', 'state', 'registration_date']
------------------------------------------------------------
products: 2000 rows
Columns: ['product_id', 'product_name', 'category', 'price']
------------------------------------------------------------
orders: 99600 rows
Columns: ['order_id', 'customer_id', 'order_date', 'status', 'total_amount']
------------------------------------------------------------
order_items: 199600 rows
Columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'price']
------------------------------------------------------------
payments: 99900 rows
Columns: ['payment_id', 'order_id', 'payment_method', 'payment_status', 'payment_date']
------------------------------------------------------------
========== QUARANTINE VALIDATION ==========
orders_quarantine: 500 rows
------------------------------------------------------------
order_items_quarantine: 400 rows
--